# Lab 11: Word Embeddings & Transformer Self-Attention

**Name:** Soren Patel

**Alpha:** 284932

In this lab, you will test the **Four Core Pillars** of word embeddings and build/inspect **Transformer Self-Attention** in PyTorch.

In [ ]:
import time
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel

torch.manual_seed(42)
print("PyTorch Version:", torch.__version__)

## Part 1: Math vs. Hardware — Benchmarking Token Lookup (25 pts)

Mathematically, looking up token indices $\mathbf{I} \in \{0, \dots, V-1\}^{B \times W}$ in an embedding matrix $\mathbf{E} \in \mathbb{R}^{V \times D}$ is equivalent to multiplying one-hot vectors by $\mathbf{E}$. In hardware, `nn.Embedding` performs a direct $O(1)$ pointer gather.

**TODO 1.1:** Complete both functions below and verify they produce identical outputs.

In [ ]:
def math_onehot_lookup(token_ids: torch.Tensor, E_weight: torch.Tensor) -> torch.Tensor:
    """
    Naive Math View: Convert integer token_ids (B, W) into float32 one-hot vectors (B, W, V)
    and multiply by the embedding matrix E_weight (V, D).
    """
    V = E_weight.shape[0]
    # TODO: 1. Create one-hot tensor of shape (B, W, V) and cast to .float()
    # TODO: 2. Multiply by E_weight using @ and return the (B, W, D) result
    pass

def systems_pointer_lookup(token_ids: torch.Tensor, embedding_layer: nn.Embedding) -> torch.Tensor:
    """
    Systems View: Use the nn.Embedding layer directly to gather rows in O(1) time.
    """
    # TODO: Pass token_ids directly into embedding_layer
    pass

# --- Benchmark & Verification ---
V, D = 50_000, 512
B, W = 8, 128  # 8 sequences of length 128

embedding_layer = nn.Embedding(num_embeddings=V, embedding_dim=D)
token_ids = torch.randint(0, V, (B, W))

# Time the Naive One-Hot GEMM
t0 = time.perf_counter()
out_math = math_onehot_lookup(token_ids, embedding_layer.weight)
t_math = (time.perf_counter() - t0) * 1000

# Time the Systems Pointer Lookup
t0 = time.perf_counter()
out_sys = systems_pointer_lookup(token_ids, embedding_layer)
t_sys = (time.perf_counter() - t0) * 1000

assert torch.allclose(out_math, out_sys, atol=1e-5), "Outputs do not match!"
onehot_megabytes = (B * W * V * 4) / (1024 ** 2)

print(f"Outputs match! Shape: {out_sys.shape}")
print(f"One-Hot Intermediate Tensor Size: {onehot_megabytes:.1f} MB of 99.998% zeros!")
print(f"Naive One-Hot GEMM Time: {t_math:.2f} ms")
print(f"Systems Pointer Gather Time: {t_sys:.3f} ms")
print(f"Hardware Speedup: {t_math / t_sys:.1f}x faster!")

**Question 1.1:** Why did `math_onehot_lookup` take so much longer and allocate over 200 MB of extra RAM for just 8 short paragraphs, even though it computed the exact same answer as `systems_pointer_lookup`?

**Your Answer:** *Write 2-3 sentences here.*

## Part 2: Navigating the Geometry of $\mathbb{R}^D$ (25 pts)

Let's load the real pre-trained token embedding table $\mathbf{E} \in \mathbb{R}^{30522 \times 768}$ from `distilbert-base-uncased` and explore its geometry using Cosine Similarity.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModel.from_pretrained("distilbert-base-uncased", output_attentions=True, output_hidden_states=True)
model.eval()

# Extract the static embedding lookup matrix E of shape (V, D) = (30522, 768)
E_matrix = model.embeddings.word_embeddings.weight.detach()
# Pre-normalize rows of E so cosine similarity is just a fast dot product!
E_norm = F.normalize(E_matrix, p=2, dim=1)
print("Loaded static embedding matrix E of shape:", E_matrix.shape)

In [ ]:
def find_nearest_tokens(word: str, top_k: int = 6):
    """
    Given a single-token word, compute cosine similarity between its normalized vector
    and all rows in E_norm, returning the top_k closest words and their scores.
    """
    token_id = tokenizer.convert_tokens_to_ids(word.lower())
    assert token_id != tokenizer.unk_token_id, f"'{word}' is not a single token in the vocabulary!"

    # TODO: 1. Grab the normalized vector for `token_id` from E_norm (shape: 768,)
    # TODO: 2. Compute cosine similarities against all words in E_norm using matrix-vector multiply (@)
    # TODO: 3. Use torch.topk(sims, k=top_k) to get top values and indices
    pass

for query_word in ["destroyer", "submarine", "cyber", "annapolis"]:
    print(f"\nTop neighbors for '{query_word}':", find_nearest_tokens(query_word, top_k=6))

In [ ]:
def solve_analogy(word_a: str, word_b: str, word_c: str, top_k: int = 4):
    """
    Solves: word_a is to word_b as word_c is to ???
    Computes target_vec = E_norm[b] - E_norm[a] + E_norm[c], normalizes it,
    and returns the top_k closest words in E_norm.
    """
    id_a = tokenizer.convert_tokens_to_ids(word_a.lower())
    id_b = tokenizer.convert_tokens_to_ids(word_b.lower())
    id_c = tokenizer.convert_tokens_to_ids(word_c.lower())

    # TODO: Compute target_vec = vec_b - vec_a + vec_c, normalize it, and find top_k nearest tokens in E_norm
    pass

print("man : king :: woman ->", solve_analogy("man", "king", "woman"))
print("paris : france :: tokyo ->", solve_analogy("paris", "france", "tokyo"))
print("army : general :: navy ->", solve_analogy("army", "general", "navy"))

## Part 3: Coding Scaled Dot-Product Self-Attention from Scratch (25 pts)

Implement the core Transformer equation:
$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{Softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_k}}\right)\mathbf{V}$$

In [ ]:
def scaled_dot_product_attention(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor):
    """
    Args:
        Q, K, V: Tensors of shape (W, d_k)
    Returns:
        H: Contextualized output vectors of shape (W, d_k)
        A: Attention weight matrix of shape (W, W) where each row sums to 1.0
    """
    d_k = Q.shape[-1]
    # TODO: 1. Compute scaled scores S = (Q @ K.T) / math.sqrt(d_k)
    # TODO: 2. Compute row-wise attention probabilities A = torch.softmax(S, dim=-1)
    # TODO: 3. Compute weighted values H = A @ V
    # TODO: Return H, A
    pass

# --- Synthetic "Semantic Tugboat" Test on ["foggy", "naval", "port"] ---
# Suppose d_k = 4, and we have 3 tokens: 0="foggy", 1="naval", 2="port"
Q_toy = torch.tensor([
    [1.0, 0.0, 0.0, 0.0],  # foggy query
    [0.0, 1.0, 0.0, 0.0],  # naval query
    [0.0, 2.5, 1.0, 0.0],  # port query (strongly looking for maritime adjectives in dim 1!)
])
K_toy = torch.tensor([
    [1.0, 0.0, 0.0, 0.0],  # foggy key
    [0.0, 2.5, 0.0, 0.0],  # naval key (advertises maritime adjective in dim 1!)
    [0.0, 0.0, 1.0, 0.0],  # port key
])
V_toy = torch.tensor([
    [1.0, 0.0, 0.0, 0.0],  # foggy value
    [0.0, 1.0, 0.0, 0.0],  # naval value (pure maritime feature)
    [0.0, 0.0, 1.0, 0.0],  # port value (initially orthogonal to naval!)
])

H_toy, A_toy = scaled_dot_product_attention(Q_toy, K_toy, V_toy)
print("Attention Weights A (rows = [foggy, naval, port]):\n", A_toy.round(decimals=3))
print("\nCosine similarity between 'port' and 'naval' BEFORE Attention:",
      F.cosine_similarity(V_toy[2], V_toy[1], dim=0).item())
print("Cosine similarity between 'port' and 'naval' AFTER Attention:",
      F.cosine_similarity(H_toy[2], V_toy[1], dim=0).item())

## Part 4: Watching `"port"` Change Meaning in a Real Transformer (25 pts)

Now let's pass two very different sentences containing `"port"` through `DistilBERT` and measure how Self-Attention pulls their vectors apart across the 6 Transformer layers!

In [ ]:
sent_a = "The destroyer anchored in the foggy naval port."
sent_b = "Plug the ethernet cable into the open network port."

inputs_a = tokenizer(sent_a, return_tensors="pt")
inputs_b = tokenizer(sent_b, return_tensors="pt")

tokens_a = tokenizer.convert_ids_to_tokens(inputs_a["input_ids"][0])
tokens_b = tokenizer.convert_ids_to_tokens(inputs_b["input_ids"][0])

idx_port_a = tokens_a.index("port")
idx_port_b = tokens_b.index("port")

with torch.no_grad():
    out_a = model(**inputs_a)
    out_b = model(**inputs_b)

# out.hidden_states is a tuple of 7 tensors: Layer 0 (static embedding) through Layer 6 (final layer)
# Each tensor has shape (1, W, 768)
# TODO: Compute the cosine similarity between the vector for "port" in Sentence A and Sentence B
# at EVERY layer from Layer 0 to Layer 6, and store them in a list `layer_similarities`.
layer_similarities = []
for layer_idx in range(len(out_a.hidden_states)):
    # TODO: Extract vec_a and vec_b for "port" at `layer_idx` and compute F.cosine_similarity
    pass

for l, sim in enumerate(layer_similarities):
    print(f"Layer {l} Cosine Similarity between naval 'port' and network 'port': {sim:.4f}")

# Plot the divergence across layers
plt.figure(figsize=(7, 4))
plt.plot(range(len(layer_similarities)), layer_similarities, marker="o", linewidth=2.5, color="#0284c7")
plt.title("Polysemy Resolution: Cosine Similarity of 'port' Across Transformer Layers")
plt.xlabel("Transformer Layer (0 = Static Lookup, 6 = Final Contextual Layer)")
plt.ylabel("Cosine Similarity (Sent A vs Sent B)")
plt.ylim(0.4, 1.05)
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

**Question 4.1:** Why is the cosine similarity between `"port"` in Sentence A and `"port"` in Sentence B nearly `1.0` at Layer 0, and why does it drop steadily as we go deeper into Layer 6?

**Your Answer:** *Write 2-3 sentences here.*: